# Tema 3 · Evaluación, Validación y Métricas (Clasificación y Regresión)
### Libreta con *scikit-learn*

**Objetivos de aprendizaje**
- Comprender por qué y cómo evaluamos modelos de ML.
- Aplicar **hold-out** y **validación cruzada** (k-fold y estratificada).
- Calcular e interpretar **métricas de clasificación** (accuracy, precision, recall, F1, balanced accuracy, ROC-AUC, PR-AUC).
- Calcular e interpretar **métricas de regresión** (MSE, RMSE, MAE, R²).
- Implementar métricas básicas *from scratch* para afianzar conceptos.

## 1️⃣ Introducción: ¿Por qué evaluar? Generalización y sobreajuste
- El objetivo es estimar la **utilidad real** del modelo en **datos no vistos**.
- El **error de entrenamiento** no es fiable: el modelo puede **sobreajustar**.
- Necesitamos procedimientos de **validación** que den estimaciones honestas del rendimiento.

## 2️⃣ Preparación del entorno y librerías
A continuación importamos las librerías necesarias. Usaremos `Pipeline` para **evitar fugas de datos** y encapsular el preprocesado.

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import make_classification, make_regression
from sklearn.model_selection import train_test_split, KFold, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import Ridge, LogisticRegression
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, balanced_accuracy_score,
    roc_auc_score, average_precision_score, confusion_matrix,
    mean_squared_error, mean_absolute_error, r2_score, fbeta_score
)

np.random.seed(42)

## 3️⃣ Conjuntos de datos sintéticos y partición *hold-out*
- **Clasificación:** datos potencialmente **desbalanceados** para mostrar la utilidad de métricas robustas.
- **Regresión:** datos con ruido controlado.
- **Hold-out:** separamos en *train/test* (clasificación estratificada).

In [3]:
# Clasificación (desbalance opcional)
Xc, yc = make_classification(
    n_samples=1000, n_features=10, n_informative=6, n_redundant=2,
    weights=[0.75, 0.25], random_state=42
)

# Regresión
Xr, yr = make_regression(
    n_samples=800, n_features=8, noise=12.0, random_state=42
)

# Hold-out (estratificado para clasificación)
Xc_tr, Xc_te, yc_tr, yc_te = train_test_split(
    Xc, yc, test_size=0.25, stratify=yc, random_state=42
)
Xr_tr, Xr_te, yr_tr, yr_te = train_test_split(
    Xr, yr, test_size=0.25, random_state=42
)

print("Clasificación:", Xc_tr.shape, Xc_te.shape, np.bincount(yc))
print("Regresión:", Xr_tr.shape, Xr_te.shape)

Clasificación: (750, 10) (250, 10) [747 253]
Regresión: (600, 8) (200, 8)


## 4️⃣ Evaluación *hold-out* en **clasificación** con métricas básicas
- Modelo: **Regresión logística** con **estandarización** (Pipeline).
- Métricas: `accuracy`, `precision`, `recall`, `F1`, `balanced accuracy`, **ROC-AUC**, **PR-AUC**.
- Matriz de confusión para interpretar errores.

In [4]:
clf = Pipeline([
    ("scaler", StandardScaler()),
    ("logreg", LogisticRegression(max_iter=1000, random_state=42))
])

clf.fit(Xc_tr, yc_tr)
proba = clf.predict_proba(Xc_te)[:, 1] # Probabilidades de la clase positiva
pred  = (proba >= 0.5).astype(int)

acc  = accuracy_score(yc_te, pred)
prec = precision_score(yc_te, pred, zero_division=0)
rec  = recall_score(yc_te, pred, zero_division=0)
f1   = f1_score(yc_te, pred, zero_division=0)
bal  = balanced_accuracy_score(yc_te, pred)
auc  = roc_auc_score(yc_te, proba)
ap   = average_precision_score(yc_te, proba)
cm   = confusion_matrix(yc_te, pred)

print(f"Hold-out — acc={acc:.3f}, prec={prec:.3f}, rec={rec:.3f}, f1={f1:.3f}, bal_acc={bal:.3f}, ROC-AUC={auc:.3f}, PR-AUC={ap:.3f}")
print("Matriz de confusión:\n", cm)

Hold-out — acc=0.812, prec=0.674, rec=0.492, f1=0.569, bal_acc=0.706, ROC-AUC=0.883, PR-AUC=0.741
Matriz de confusión:
 [[172  15]
 [ 32  31]]


## 5️⃣ Validación cruzada (k-fold y estratificada) en **clasificación**
- **k-fold**: promedia el rendimiento sobre *k* particiones.
- **Estratificada**: mantiene proporciones de clase en cada fold (importante en desbalance).

In [5]:
k = 5
clf_cv = Pipeline([
    ("scaler", StandardScaler()),
    ("logreg", LogisticRegression(max_iter=1000, random_state=42))
])

kf  = KFold(n_splits=k, shuffle=True, random_state=42)
skf = StratifiedKFold(n_splits=k, shuffle=True, random_state=42)

scores_kf  = cross_val_score(clf_cv, Xc, yc, scoring="accuracy", cv=kf)
scores_skf = cross_val_score(clf_cv, Xc, yc, scoring="accuracy", cv=skf)

print(f"KF acc: mean={scores_kf.mean():.3f} ± {scores_kf.std():.3f}")
print(f"SKFold acc: mean={scores_skf.mean():.3f} ± {scores_skf.std():.3f}")

KF acc: mean=0.835 ± 0.018
SKFold acc: mean=0.838 ± 0.017


## 6️⃣ Evaluación *hold-out* en **regresión** con métricas clave
- Modelo: **Ridge** con **estandarización** (Pipeline).
- Métricas: **MSE**, **RMSE**, **MAE**, **R²**.

In [6]:
reg = Pipeline([
    ("scaler", StandardScaler()),
    ("ridge", Ridge(alpha=1.0, random_state=42))
])

reg.fit(Xr_tr, yr_tr)
yr_pred = reg.predict(Xr_te)

mse  = mean_squared_error(yr_te, yr_pred)
rmse = np.sqrt(mse)
mae  = mean_absolute_error(yr_te, yr_pred)
r2   = r2_score(yr_te, yr_pred)

print(f"Hold-out — MSE={mse:.2f}, RMSE={rmse:.2f}, MAE={mae:.2f}, R2={r2:.3f}")

Hold-out — MSE=175.27, RMSE=13.24, MAE=10.62, R2=0.991


## 7️⃣ Validación cruzada en **regresión**
- Repite el entrenamiento y evaluación con distintas particiones para obtener una media más estable del rendimiento.

In [7]:
reg_cv = Pipeline([
    ("scaler", StandardScaler()),
    ("ridge", Ridge(alpha=1.0, random_state=42))
])

scores_r2 = cross_val_score(reg_cv, Xr, yr, scoring="r2", cv=KFold(n_splits=5, shuffle=True, random_state=42))
print(f"R2 CV: mean={scores_r2.mean():.3f} ± {scores_r2.std():.3f}")

R2 CV: mean=0.992 ± 0.001


## 8️⃣ Implementación *from scratch* de métricas básicas
Para afianzar conceptos, implementamos manualmente algunas métricas y verificamos con `sklearn.metrics`.

In [ ]:
def my_accuracy(y_true, y_pred):
    # TODO: completar
    TP = np.sum((y_pred == 1) & (y_true == 1))
    TN = np.sum((y_pred == 0) & (y_true == 0))
    return (TP + TN) / len(y_true)

def my_precision(y_true, y_pred):
    # TODO: completar
    TP = np.sum((y_pred == 1) & (y_true == 1))
    FP = np.sum((y_pred == 1) & (y_true == 0))
    return TP / (TP + FP)

def my_recall(y_true, y_pred):
    TP = np.sum((y_pred == 1) & (y_true == 1))
    FN = np.sum((y_pred == 0) & (y_true == 1))
    return TP/(TP+FN)

def my_f1(y_true, y_pred):
    prec = my_precision(y_true, y_pred)
    rec = my_recall(y_true, y_pred)
    return 2*(prec*rec)/(prec+rec)

def my_beta_f1(y_true, y_pred, beta=1.0):
    prec = my_precision(y_true, y_pred)
    rec = my_recall(y_true, y_pred)
    return (1 + beta**2) * (prec * rec) / ((beta**2 * prec) + rec)

# Verificación
y_true_c = np.array([0,1,1,0,1,0,0,1])
y_pred_c = np.array([0,1,0,0,1,0,1,1])
assert np.isclose(my_beta_f1(y_true_c, y_pred_c), fbeta_score(y_true_c, y_pred_c, beta=1.0, zero_division=0))
assert np.isclose(my_accuracy(y_true_c, y_pred_c), accuracy_score(y_true_c, y_pred_c))
assert np.isclose(my_precision(y_true_c, y_pred_c), precision_score(y_true_c, y_pred_c, zero_division=0))
assert np.isclose(my_recall(y_true_c, y_pred_c), recall_score(y_true_c, y_pred_c, zero_division=0))
assert np.isclose(my_f1(y_true_c, y_pred_c), f1_score(y_true_c, y_pred_c, zero_division=0))

print("Verificaciones from-scratch OK")

Verificaciones from-scratch OK


## 9️⃣ Implementación *from scratch* de métricas y validaciones
Para afianzar conceptos, implementamos manualmente algunas tecnicas de validación `sklearn.metrics`.

In [25]:
def my_balanced_accuracy(y_true, y_pred):
    #TODO: completar
    TP = np.sum((y_pred == 1) & (y_true == 1))
    TN = np.sum((y_pred == 0) & (y_true == 0))
    FN = np.sum((y_pred == 0) & (y_true == 1))
    FP = np.sum((y_pred == 1) & (y_true == 0)) 
    TPR = TP/(TP+FN)
    TNR = TN/(TN+FP)
    return (TPR+TNR)/2



# Verificación Balanced Accuracy
y_true_c = np.array([0,1,1,0,1,0,0,1])
y_pred_c = np.array([0,1,0,0,1,0,1,1])
assert np.isclose(my_balanced_accuracy(y_true_c, y_pred_c), balanced_accuracy_score(y_true_c, y_pred_c))
print("Verificación Balanced Accuracy from-scratch OK")


def my_roc_auc(y_true, y_scores): # Implementación básica de ROC-AUC from-scratch (usando la regla del trapecio).
    # y_true: etiquetas verdaderas (0 o 1)
    # y_scores: scores de predicción (valores continuos)
    # Ordenar por scores
    desc_score_indices = np.argsort(-y_scores)  # Indices para ordenar de mayor a menor
    y_true = y_true[desc_score_indices]  # Etiquetas verdaderas ordenadas
    y_scores = y_scores[desc_score_indices]  # Scores ordenados

    # Calcular TPR y FPR
    # TODO: completar
    tpr = np.cumsum(y_true == 1)/np.sum(y_true == 1)
    fpr = np.cumsum(y_true == 0)/np.sum(y_true == 0)
    

    # Añadir punto (0,0)
    tpr = np.concatenate([[0], tpr])
    fpr = np.concatenate([[0], fpr])

    # Calcular AUC usando la regla del trapecio
    # TODO: completar
    h = np.diff(fpr)
    auc = np.sum(h * (tpr[1:] + tpr[:-1]) / 2)
    return auc



# Verificación ROC-AUC
y_true_c = np.array([0,1,1,0,1,0,0,1])
y_scores_c = np.array([0.1,0.9,0.8,0.4,0.95,0.2,0.3,0.85])
assert np.isclose(my_roc_auc(y_true_c, y_scores_c), roc_auc_score(y_true_c, y_scores_c))
print("Verificación ROC-AUC from-scratch OK")


def my_cv_validation(X, y, model, cv): # Implementación básica de validación cruzada from-scratch, el parámetro cv es un objeto de validación cruzada (e.g., KFold, StratifiedKFold):
    # X: características
    # y: etiquetas
    # model: modelo con métodos fit y predict
    # cv: objeto de validación cruzada (e.g., KFold, StratifiedKFold)
    scores = []
    for train_idx, val_idx in cv.split(X, y):
        X_train, X_val = X[train_idx], X[val_idx]
        y_train, y_val = y[train_idx], y[val_idx]
        model.fit(X_train, y_train)
        y_pred = model.predict(X_val)
        score = accuracy_score(y_val, y_pred)  # Usamos accuracy como métrica
        scores.append(score)
    return np.array(scores)


# Verificación Validación Cruzada
Xc, yc = make_classification(n_samples=100, n_features=5, random_state=42)
model = LogisticRegression(max_iter=1000, random_state=42)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scores = my_cv_validation(Xc, yc, model, cv)    
assert np.allclose(scores, cross_val_score(model, Xc, yc, scoring="accuracy", cv=cv))
print("Verificación Validación Cruzada from-scratch OK")

Verificación Balanced Accuracy from-scratch OK
Verificación ROC-AUC from-scratch OK
Verificación Validación Cruzada from-scratch OK


## 🔟 Implementación *from scratch* de particiones de train y test en una k-fold cv
Para afianzar conceptos, implementamos manualmente algunas tecnicas de validación.

In [26]:
def my_cv_split(X, y, n_kf, random_state=None):
    """
    Implementación de K-Fold cross-validation desde cero.
    
    Parameters:
    -----------
    X : array-like
        Características
    y : array-like
        Etiquetas
    n_kf : int
        Número de folds
    random_state : int, optional
        Semilla para reproducibilidad
        
    Yields:
    -------
    train_idx : array
        Índices para entrenamiento
    test_idx : array
        Índices para prueba
    """
    n_samples = X.shape[0]
    indices = np.arange(n_samples)
    
    if random_state is not None:
        np.random.seed(random_state)
    np.random.shuffle(indices)

    # Calcular tamaños de cada fold
    fold_sizes = np.full(n_kf, n_samples // n_kf, dtype=int)
    fold_sizes[:n_samples % n_kf] += 1  # Distribuir el resto
    
    
    # Crear los límites de cada fold TODO
    # Generar splits: cada fold es test una vez TODO completar
    actual = 0
    for i in range(n_kf):
        inicio = actual
        fin = inicio + fold_sizes[i]
        t = indices[inicio:fin]
        tr = np.concatenate([indices[:inicio], indices[fin:]])
        actual = fin
        yield tr, t
    
    

# Verificación básica K-Fold
print("="*60)
print("VERIFICACIÓN K-FOLD FROM-SCRATCH")
print("="*60 + "\n")

Xc, yc = make_classification(n_samples=100, n_features=5, random_state=42)

kf = my_cv_split(Xc, yc, 5, random_state=42)

for i, (train_idx, test_idx) in enumerate(kf):
    print(f"\n--- Fold {i+1} ---")
    print(f"Train indices (primeros 10): {train_idx[:10]}")
    print(f"Test indices: {test_idx}")
    print(f"Train size: {len(train_idx)}, Test size: {len(test_idx)}")
    
    # Verificar que no hay overlap
    assert len(np.intersect1d(train_idx, test_idx)) == 0, "¡Hay overlap entre train y test!"
    # Verificar que juntos cubren todos los datos
    assert len(np.union1d(train_idx, test_idx)) == len(Xc), "¡No cubren todos los datos!"

print("\n" + "="*60)
print("✓ Verificación K-Fold from-scratch OK")
print("="*60)


# Comparación con sklearn
print("\n" + "="*60)
print("COMPARACIÓN CON SKLEARN")
print("="*60 + "\n")

sklearn_kf = KFold(n_splits=5, shuffle=True, random_state=42)

print("Estructura de splits (sklearn):")
for i, (train_idx, test_idx) in enumerate(sklearn_kf.split(Xc)):
    print(f"Fold {i+1}: Train size={len(train_idx)}, Test size={len(test_idx)}")

VERIFICACIÓN K-FOLD FROM-SCRATCH


--- Fold 1 ---
Train indices (primeros 10): [55 88 26 42 69 15 40 96  9 72]
Test indices: [83 53 70 45 44 39 22 80 10  0 18 30 73 33 90  4 76 77 12 31]
Train size: 80, Test size: 20

--- Fold 2 ---
Train indices (primeros 10): [83 53 70 45 44 39 22 80 10  0]
Test indices: [55 88 26 42 69 15 40 96  9 72 11 47 85 28 93  5 66 65 35 16]
Train size: 80, Test size: 20

--- Fold 3 ---
Train indices (primeros 10): [83 53 70 45 44 39 22 80 10  0]
Test indices: [49 34  7 95 27 19 81 25 62 13 24  3 17 38  8 78  6 64 36 89]
Train size: 80, Test size: 20

--- Fold 4 ---
Train indices (primeros 10): [83 53 70 45 44 39 22 80 10  0]
Test indices: [56 99 54 43 50 67 46 68 61 97 79 41 58 48 98 57 75 32 94 59]
Train size: 80, Test size: 20

--- Fold 5 ---
Train indices (primeros 10): [83 53 70 45 44 39 22 80 10  0]
Test indices: [63 84 37 29  1 52 21  2 23 87 91 74 86 82 20 60 71 14 92 51]
Train size: 80, Test size: 20

✓ Verificación K-Fold from-scratch OK

COMPARACIÓN